# P1 — Task 1 — EDA + slide Giới thiệu

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task1_eda.ipynb)

**Slide sở hữu:** 4, 5, 6, 7 · **Slide mới:** +1: Q3 kênh theo tỉnh  
**Đã có:** task1_eda.py, task1_findings.md, chart t1_monthly_spend / t1_essential_discretionary / t1_category_ticket / t1_age_cohort

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, sys, gzip, shutil
DRIVE_DATA = "/content/drive/MyDrive/BI10_ROUND01_DATASET"   # thư mục data trên Drive (xem sheet 'Setup Colab')
REPO = "/content/BI10GroupYAPPERS"

from google.colab import drive, userdata
drive.mount("/content/drive")

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f in ["consumer_financial_health_engagement_2025.csv", "data_dictionary.xlsx"]:
    if not os.path.exists(f"{D}/{f}") and os.path.exists(f"{DRIVE_DATA}/{f}"):
        shutil.copy(f"{DRIVE_DATA}/{f}", f"{D}/{f}")
if not os.path.exists(TX):
    gz, raw = f"{DRIVE_DATA}/consumer_transactions_2025.parquet.gz", f"{DRIVE_DATA}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
    if os.path.exists(gz):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
        with gzip.open(gz) as src, open(TX, "wb") as dst: shutil.copyfileobj(src, dst)
    else:                    # hoặc upload nguyên thư mục .parquet/ chứa CSV 597MB
        shutil.copy(raw, TX)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task1_output.txt`.

In [ ]:
%run -i draft/task1_eda.py

## Việc 1 — Q3 — bảng kênh chi tiết theo tỉnh (SLIDE MỚI)  `[BẮT BUỘC · hạn 27/09 12:00]`
Đề bắt 'compare channel breakdown (POS vs digital)' cho ≥2 tỉnh. Với Hà Nội, Đồng Nai, Lâm Đồng, Hưng Yên, Hải Phòng + toàn quốc: tỷ trọng SỐ GIAO DỊCH và GIÁ TRỊ theo POS / QR / E-com / Mobile / Recurring. Chart stacked bar 100% ngang (6 thanh). Kết luận giữ: chênh ±0.5pp → địa lý không phải đòn bẩy.

**Giao:** Chart t1_province_channel.png + bullet slide mới trong slide_outline.md

In [ ]:
# Gợi ý:
# pd.crosstab(txn.province_city, txn.transaction_channel, values=txn.spend_amount_vnd, aggfunc='sum', normalize='index')


## Việc 2 — Q2 — chứng minh độ dốc, không chỉ 2 đầu  `[Cao · hạn 27/09 12:00]`
Chia FHS thành bin 10 điểm, tính tỷ trọng discretionary từng bin → line chart. Đường giảm đều = 'đảo chiều liên tục, không do ngưỡng 40/80'. Thử thêm ngưỡng FHS<45 vs ≥75 để chứng minh độ bền.

**Giao:** Line chart + 1 bullet slide 6

In [ ]:
# Gợi ý:
# mon['bin']=pd.cut(mon.financial_health_score, range(0,101,10)); mon.groupby('bin').discretionary_spend_ratio.mean()


## Việc 3 — Q1 — tháng 12 tăng vì category nào?  `[Trung bình · hạn 27/09 18:00]`
Mức tăng số giao dịch T12 vs T2 theo category → top 5 đóng góp. Kiểm tra caveat 'gộp 2 năm vào 2025' bằng timestamp gốc (có năm 2024 không?) và ghi rõ kết quả.

**Giao:** 2 dòng findings + bullet slide 5

In [ ]:
# Gợi ý:
# g=txn.groupby(['transaction_month','spending_category']).size().unstack(); (g.loc[12]-g.loc[2]).nlargest(5)


## Việc 4 — Q4 — bổ sung chỉ số thói quen  `[Thấp · hạn 27/09 18:00]`
% khách có mua Fuel / Groceries; số lần mua trung bình / khách / tháng cho 2 category top.

**Giao:** Bullet slide 7

In [ ]:
# Gợi ý:
# txn[txn.spending_category==CAT].groupby(['consumer_id','transaction_month']).size().mean()


## Việc 5 — Q5 — tuổi là tín hiệu yếu (bootstrap CI 95%)  `[Trung bình · hạn 27/09 18:00]`
Bootstrap 1000 lần FHS trung bình từng nhóm tuổi (giữ 7 khách 15–17 tuổi). Nếu các CI chồng nhau → nói rõ 'tuổi không phân biệt được'. Gửi bảng CI cho P2 dùng chung.

**Giao:** Bảng CI (gửi P2) + bullet slide 7

In [ ]:
# Gợi ý:
# np.percentile([s.sample(frac=1,replace=True).mean() for _ in range(1000)], [2.5,97.5])


## Việc 6 — Slide 4 — rà số dataset  `[Thấp · hạn 27/09 18:00]`
Kiểm lại 999 khách / 1,852,394 giao dịch / 10,992 consumer-month và 2 luật (tỷ lệ, wellbeing).

**Giao:** Xác nhận cho P5

In [ ]:
# Gợi ý:
# len(mon), mon.consumer_id.nunique(), len(txn)


## Findings mới (2–4 dòng / việc, có số)
- ...

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task1_eda.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.